# データ工程: PDFの画像化とローカルVLM（Qwen2.5-VL）による構造化テキスト抽出

このノートブックでは、DGX Sparkのリソースを活用し、以下の処理を行います：
1. 依存パッケージのインストール
2. `row_data` フォルダ内のPDFを画像に変換し、`extract_data` に保存
3. ローカルで動作する最新のVLM（Qwen2.5-VL）を使用して構造化テキストの抽出

## 1. 依存パッケージのインストール

In [3]:
# システム依存パッケージ (poppler-utils) のインストール
!apt-get update && apt-get install -y poppler-utils

# Pythonライブラリのインストール (Qwen2.5-VL対応版)
!pip install pdf2image transformers accelerate qwen-vl-utils ipywidgets python-dotenv pymupdf

!pip install openai






Get:1 http://ports.ubuntu.com/ubuntu-ports noble InRelease [256 kB]
Get:2 http://ports.ubuntu.com/ubuntu-ports noble-updates InRelease [126 kB]
Get:3 http://ports.ubuntu.com/ubuntu-ports noble-backports InRelease [126 kB]
Get:4 http://ports.ubuntu.com/ubuntu-ports noble-security InRelease [126 kB]
Get:5 http://ports.ubuntu.com/ubuntu-ports noble/universe arm64 Packages [19.0 MB]
Get:6 http://ports.ubuntu.com/ubuntu-ports noble/main arm64 Packages [1776 kB] 
Get:7 http://ports.ubuntu.com/ubuntu-ports noble/restricted arm64 Packages [113 kB]
Get:8 http://ports.ubuntu.com/ubuntu-ports noble/multiverse arm64 Packages [274 kB]
Get:9 http://ports.ubuntu.com/ubuntu-ports noble-updates/restricted arm64 Packages [5004 kB]
Get:10 http://ports.ubuntu.com/ubuntu-ports noble-updates/multiverse arm64 Packages [43.7 kB]
Get:11 http://ports.ubuntu.com/ubuntu-ports noble-updates/main arm64 Packages [2445 kB]
Get:12 http://ports.ubuntu.com/ubuntu-ports noble-updates/universe arm64 Packages [1987 kB]
Get

## 2. ライブラリのインポートと設定

In [4]:
import torch
import os
from pathlib import Path
from pdf2image import convert_from_path
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
from PIL import Image
import IPython.display
from dotenv import load_dotenv

load_dotenv() # .envファイルを読み込む

# パスの設定
current_dir = Path(os.getcwd())
print(f"Current working directory: {current_dir}")

# row_data フォルダがノートブックの場所にあると想定されるため、探索パスを定義します
search_paths = [
    # 1. コンテナ内での絶対パスの候補
    Path("/workspace/01_training/01_data_prep"),
    # 2. カレントディレクトリの隣にある 01_training を探す場合（/workspace/container にいる場合）
    current_dir.parent / "01_training" / "01_data_prep",
    # 3. カレントディレクトリ内の場合
    current_dir
]

BASE_DIR = None
for p in search_paths:
    if (p / "row_data").exists():
        BASE_DIR = p
        print(f"Found row_data in: {BASE_DIR.absolute()}")
        break

if BASE_DIR is None:
    print("Warning: 'row_data' directory not found in typical locations. Using Current working directory.")
    BASE_DIR = current_dir

ROW_DATA_DIR = BASE_DIR / "row_data"
EXTRACT_DATA_DIR = BASE_DIR / "extract_data"

print(f"ROW_DATA_DIR: {ROW_DATA_DIR.absolute()}")
print(f"EXTRACT_DATA_DIR: {EXTRACT_DATA_DIR.absolute()}")

EXTRACT_DATA_DIR.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Current working directory: /home/hiroooo000-remote/dev/ft-test/01_training/02_SFT
ROW_DATA_DIR: /home/hiroooo000-remote/dev/ft-test/01_training/02_SFT/row_data
EXTRACT_DATA_DIR: /home/hiroooo000-remote/dev/ft-test/01_training/02_SFT/extract_data
Using device: cuda


## 2.2 Huggingfaceへのログイン

In [3]:
from huggingface_hub import login
# これを実行すると、トークンを入力するテキストボックスが表示されます
login()


## 3. PDFから画像への変換

### 3.1 関数定義

In [ ]:
def convert_pdfs_to_images(row_data_path, output_path):
    """
    row_data_path 内のすべてのPDFを全ページ画像化します。
    """
    row_data_path = Path(row_data_path)
    output_path = Path(output_path)
    
    # Linux環境の大文字小文字を考慮して探索 (.pdf と .PDF)
    pdf_files = sorted(list(row_data_path.glob("*.pdf")) + list(row_data_path.glob("*.PDF")))
    
    print(f"Searching in: {row_data_path.absolute()}")
    print(f"Found {len(pdf_files)} PDF files.")
    
    if not pdf_files:
        # フォルダが存在するか確認し、存在する場合は中身をデバッグ表示
        if row_data_path.exists():
            print(f"Directory exists, files in it: {[f.name for f in row_data_path.iterdir()]}")
        else:
            print(f"Error: Directory {row_data_path} does not exist.")
        return
    
    for pdf_file in pdf_files:
        print(f"Converting {pdf_file.name}...")
        try:
            images = convert_from_path(str(pdf_file))
            for i, image in enumerate(images):
                image_name = f"{pdf_file.stem}_page_{i+1:03d}.png"
                image.save(output_path / image_name, "PNG")
                print(f" Saved: {image_name}")
        except Exception as e:
            print(f" Error converting {pdf_file.name}: {e}")

### 3.2 実行

In [ ]:
convert_pdfs_to_images(ROW_DATA_DIR, EXTRACT_DATA_DIR)

## 4. ローカルVLM（Qwen2.5-VL）によるテキスト抽出

### 4.1 関数定義

In [4]:
def load_model_qwen2_5_vl(model_name="Qwen/Qwen2.5-VL-7B-Instruct"):
    """
    Qwen2.5-VL モデルとプロセッサをロードします。
    """
    print(f"Loading model {model_name}...")
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        model_name, torch_dtype="auto", device_map="auto"
    )
    processor = AutoProcessor.from_pretrained(model_name)
    return model, processor

def load_model_cosmos_vl(model_name="nvidia/Cosmos-Reason2-8B"):
    """
    Cosmos-Reason2-8B モデルとプロセッサをロードします。
    """
    print(f"Loading model {model_name}...")
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        model_name, torch_dtype="auto", device_map="auto"
    )
    processor = AutoProcessor.from_pretrained(model_name)
    return model, processor

In [5]:
def extract_text_locally(image_path, model, processor, prompt):
    """
    ローカルのQwen2.5-VLを使用して画像から構造化テキストを抽出します。
    """
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": str(image_path),
                },
                {"type": "text", "text": prompt},
            ],
        }
    ]

    # プロンプトの準備
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )
    inputs = inputs.to(device)

    # 推論の実行
    generated_ids = model.generate(**inputs, max_new_tokens=4096)
    generated_ids_trimmed = [
        out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    return output_text[0]

### 4.2 モデルのロード

In [10]:
# モデルのロード（初回実行時はダウンロードに時間がかかります）
model, processor = load_model_cosmos_vl()

### 4.3 推論の実行

In [11]:
prompt = """
この画像のテキストを、レイアウト（見出し、表、箇条書き）を可能な限り保持してマークダウン形式で抽出してください。
「この形式でテキストが抽出されました。」の文言は不要です。
"""




In [12]:
# 最初の1ページを処理
sample_images = sorted(list(EXTRACT_DATA_DIR.glob("*.png")))
if sample_images:
    sample_path = sample_images[13]
    print(f"Processing: {sample_path.name}")
    extracted_content = extract_text_locally(sample_path, model, processor, prompt)
    print("\n--- Extracted Content ---")
    print(extracted_content)
else:
    print("No images found. Please run Step 3 (Execution) first.")

### 4.3 全ファイルバッチ推論

In [ ]:
import time

# Create directory to save extracted texts
EXTRACTED_TEXTS_DIR = BASE_DIR / "extracted_texts"
EXTRACTED_TEXTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Output directory: {EXTRACTED_TEXTS_DIR}")

# Get all converted images
all_images = sorted(list(EXTRACT_DATA_DIR.glob("*.png")))
total_images = len(all_images)
print(f"Found {total_images} images to process.")

if total_images > 0:
    total_start_time = time.time()
    
    for i, image_path in enumerate(all_images):
        output_file = EXTRACTED_TEXTS_DIR / f"{image_path.stem}.md"
        
        # Skip if already extracted
        if output_file.exists():
            print(f"[{i+1}/{total_images}] Skipping (already exists): {output_file.name}")
            continue
            
        print(f"[{i+1}/{total_images}] Processing: {image_path.name} ...")
        
        try:
            start_time = time.time() # Start time per file
            
            # Execute text extraction
            extracted_content = extract_text_locally(image_path, model, processor, prompt)
            
            # Save as Markdown file
            with open(output_file, "w", encoding="utf-8") as f:
                f.write(extracted_content)
                
            elapsed_time = time.time() - start_time # Time taken per file
            
            # Output processing time clearly
            print(f"  -> Time taken: {elapsed_time:.2f}s (Saved to {output_file.name})")
            
        except Exception as e:
            print(f"  -> Error processing {image_path.name}: {e}")
            
    total_elapsed_time = time.time() - total_start_time # Total time
    print(f"\nAll processing completed! Total time taken: {total_elapsed_time:.2f}s")
    
else:
    print("No images found to process. Please check your EXTRACT_DATA_DIR.")


### 4.4 モデルのアンロード

In [ ]:
import gc
import torch

# 1. 変数（モデルとプロセッサ）を削除
if 'model' in globals():
    del model
if 'processor' in globals():
    del processor

# 2. Pythonのガベージコレクションを実行してシステムのRAMを解放
gc.collect()

# 3. PyTorchのキャッシュをクリアしてGPUのVRAMを解放
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    
    # 念のため、現在使わされているGPUメモリ量をGB単位で表示
    allocated = torch.cuda.memory_allocated() / (1024 ** 3)
    reserved = torch.cuda.memory_reserved() / (1024 ** 3)
    print(f"GPU Memory Allocated: {allocated:.2f} GB")
    print(f"GPU Memory Reserved:  {reserved:.2f} GB")

print("VLMモデルをアンロードし、メモリを解放しました！")


### 4 alternative pymupdfによるテキスト抽出

In [ ]:
# ---------------------------------------------------------
# 5. [Alternative] Pythonライブラリ(PyMuPDF)による高速テキスト抽出（ページ分割版）
# ---------------------------------------------------------


import time
import fitz  # PyMuPDF
from pathlib import Path

# テキスト保存先ディレクトリの作成（VLM抽出結果とは別フォルダに保存します）
EXTRACTED_PY_DIR = BASE_DIR / "extracted_texts_python"
EXTRACTED_PY_DIR.mkdir(parents=True, exist_ok=True)

print(f"Output directory (Python Library): {EXTRACTED_PY_DIR}")

# 処理対象の元PDFファイルを取得
pdf_files = sorted(list(ROW_DATA_DIR.glob("*.pdf")) + list(ROW_DATA_DIR.glob("*.PDF")))
total_pdfs = len(pdf_files)
print(f"Found {total_pdfs} PDF files to process.")

if total_pdfs > 0:
    total_start_time = time.time()
    
    for pdf_path in pdf_files:
        print(f"Processing PDF: {pdf_path.name} ...")
        
        try:
            pdf_start_time = time.time()
            
            # PDFを開いてページごとに処理
            with fitz.open(pdf_path) as doc:
                total_pages = len(doc)
                
                for page_num in range(total_pages):
                    # ページ番号を3桁（001, 002...）でフォーマット
                    page_str = f"{page_num + 1:03d}"
                    output_file = EXTRACTED_PY_DIR / f"{pdf_path.stem}_page_{page_str}.md"
                    
                    if output_file.exists():
                        print(f"  [{page_num+1}/{total_pages}] Skipping: {output_file.name}")
                        continue
                        
                    page = doc[page_num]
                    # PDFに埋め込まれているテキストを抽出
                    text = page.get_text("text") 
                    
                    # Markdownファイルとして保存
                    with open(output_file, "w", encoding="utf-8") as f:
                        f.write(text.strip())
                        
                    print(f"  [{page_num+1}/{total_pages}] Saved: {output_file.name}")
                    
            elapsed_time = time.time() - pdf_start_time
            print(f"-> Finished {pdf_path.name} in {elapsed_time:.2f}s\n")
            
        except Exception as e:
            print(f"-> Error processing {pdf_path.name}: {e}\n")
            
    total_elapsed_time = time.time() - total_start_time
    print(f"All processing completed! Total time taken: {total_elapsed_time:.2f}s")
else:
    print("No PDFs found to process in ROW_DATA_DIR.")


# 5. データ評価


### 5.1 関数定義


In [5]:
from openai import OpenAI

def run_lmstudio_inference(system_prompt, user_text):
    """
    LM Studioのローカルサーバー経由で推論を行う関数
    """
    # LM StudioのデフォルトURL（ポート1234）を指定してクライアントを初期化
    # ⚠️ 【重要】Jupyter(DGXサーバー側)から手元のPCにあるLM Studioにアクセスする場合、
    # localhost の部分を手元のPCのIPアドレス（例: "http://192.168.x.x:1234/v1" 等）に
    # 変更するか、ローカルホストへのSSHポートフォワーディング設定が必要です。
    client = OpenAI(
        base_url="http://127.0.0.1:1234/v1", 
        api_key="lm-studio" # LM StudioではAPIキーの検証はないため任意の文字列でOK
    )

    # 送信するメッセージの構成
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_text}
    ]

    print("Generating response via LM Studio...")
    
    # サーバーにリクエストを送信
    completion = client.chat.completions.create(
        model="local-model", # 何を指定してもLM Studio側でロード中のモデルが使われます
        messages=messages,
        temperature=0.6,
        )
    return completion


In [23]:
system_prompt = """あなたはLLMの学習データ作成のエキスパートです。与えられたテキストをLLMの継続事前学習に適した形に変換してください。
"""

user_text = """
```markdown
# 障害程度別対象事業一覧表 (〇対象・△一部対象)

| 事業 | 障害の種別 | 本文頁 | 所得制限 | 身体障害者手帳 |
| --- | --- | --- | --- | --- |
|  |  |  |  | 視覚障害 | 聴覚又は平衡感覚機能障害 |
|  |  |  |  | 1 | 2 | 3 | 4 | 2 | 3 | 4 |
| 巡回入浴サービス | 82 |  |  |  |  |  |  |  |  |  |
| 軽度障害者入浴 | 82 |  |  |  |  |  |  | 本文参照 |  |  |
| 障害者・児移動支援事業 | 82 |  | 〇 | 〇 |  |  |  |  |  |  |
| 重度脳性麻痺者介護事業 | 83 |  |  |  |  |  |  |  |  |  |
| 在宅心身障害者・児緊急一時介護委託費助成 | 83 |  |  |  |  |  |  |  |  |  |
| 心身障害者(児)短期保護事業 | 84 |  | 〇 | 〇 | 〇 |  |  | 〇 | 〇 |  |
| 医療的ケア児在宅レスパイト事業 | 85 |  |  |  |  |  |  | 本文参照 |  |  |
| 居宅訪問型保育事業 | 87 |  |  |  |  |  |  | 本文参照 |  |  |
| 在宅重症心身障害児(者)等訪問事業 | 87 |  |  |  |  |  |  | 本文参照 |  |  |
| 地域安心生活支援事業 | 88 |  |  |  |  |  |  |  |  |  |
| ごみの訪問収集 | 91 |  |  |  |  |  |  | 本文参照 |  |  |
| 災害・緊急時 |  |  |  |  |  |  |  |  |  |  |
| 救急代理通報システム | 92 |  | △ | △ |  |  |  |  | △ | △ |  |
| 避難行動要支援者名簿 | 94 |  | 〇 | 〇 | 〇 | 〇 | 〇 | 〇 | 〇 | 〇 |  |
| 家具転倒防止器具の設置 | 98 |  | 〇 | 〇 | 〇 | 〇 | 〇 | 〇 | 〇 | 〇 |  |
| 情報支援 |  |  |  |  |  |  |  |  |  |  |  |
| 手話通訳者の派遣・要約筆記者の派遣 | 100 |  |  |  |  |  |  |  | △ | △ | △ |  |
| 手話通訳者の設置 | 100 |  |  |  |  |  |  |  | △ | △ | △ |  |
| 点字図書の給付 | 101 |  | △ | △ | △ | △ | △ | △ |  |  |  |
| 点字・声の広報 | 101 |  | 〇 | 〇 | 〇 | 〇 | 〇 | 〇 |  |  |  |
| 手話通訳による本議会の傍聴 | 102 |  |  |  |  |  |  | 〇 | 〇 | 〇 |  |
| 手話通訳によるケーブルテレビ(CATV)番組の視聴 | 102 |  |  |  |  |  |  | 〇 | 〇 | 〇 |  |
| 音声誘導装置の設置 | 103 |  | △ | △ | △ | △ | △ | △ |  |  |  |
| 磁気ループの設置 | 104 |  |  |  |  |  |  |  | △ | △ | △ |  |
| タクシー・自動車・駐車場・駐輪場 |  |  |  |  |  |  |  |  |  |  |  |
| 福祉タクシー・自動車燃料費助成 | 108 | 有 | 〇 | 〇 |  |  |  |  | △ |  |  |
| リフト付福祉タクシー | 109 |  |  |  |  |  |  | 本文参照 |  |  |  |
| 福祉車両の貸出 | 110 |  |  |  |  |  |  | 本文参照 |  |  |  |
| 心身障害者自動車運転免許取得経費補助 | 110 | 有 | △ | △ | △ |  |  |  | △ | △ |  |
| 身体障害者自動車改造費助成 | 110 | 有 |  |  |  |  |  |  |  |  |  |
| 駐車禁止の対象除外 | 111 |  | 〇 | 〇 | 〇 | △ |  |  | 〇 | 〇 |  |
| 駐車料金の減額 | 113 |  | 〇 | 〇 | 〇 | 〇 | 〇 | 〇 | 〇 | 〇 | 〇 |  |
| 定期利用制自転車駐車場使用料の減額 | 113 |  | 〇 | 〇 | 〇 | 〇 | 〇 | 〇 | 〇 | 〇 | 〇 |  |
| 各種料金の割引 |  |  |  |  |  |  |  |  |  |  |  |
| 都営交通の無料乗車券 | 115 |  | 〇 | 〇 | 〇 | 〇 | 〇 | 〇 | 〇 | 〇 | 〇 |  |
| 都営交通の介護者割引 | 117 |  | 〇 | 〇 | 〇 | 〇 | 〇 | 〇 | 〇 | 〇 | 〇 |  |
| 民営バス運賃の割引(身体障害者・知的障害者) | 117 |  | 〇 | 〇 | 〇 | 〇 | 〇 | 〇 | 〇 | 〇 | 〇 |  |
| 民営バス運賃の割引(精神障害者) | 117 |  |  |  |  |  |  |  |  |  |  |  |
| 有料道路通行料金の割引 | 118 |  |  |  |  |  |  | 本文参照 |  |  |  |  |
| JR線旅客運賃の割引 | 119 |  | 〇 | 〇 | 〇 | 〇 | 〇 | 〇 | 〇 | 〇 | 〇 |  |
```
"""

In [26]:
res =run_lmstudio_inference(system_prompt,user_text)

Generating response via LM Studio...


In [27]:
res

ChatCompletion(id='chatcmpl-wnv52z1cnhc3zihaeisjt4', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='\n以下は、与えられたマークダウン表を **LLM 継続事前学習向けにトークナイズしやすい構造化テキスト** に変換した例です。  \n- XML タグでテーブル・ヘッダー・行・セルを明示的に示すことで、トークン化時に「<table>」や「</tr>」などの特別トークンが自然に出現します。  \n- 空欄や「〇」「△」などの記号はそのまま残すので、元データの意味を失いません。  \n- 文末に `<EOS>`（シーケンス終端）を付与すれば、学習データの区切りとしても利用できます。  \n\n```xml\n<BOS>                     <!-- Begin Of Sequence -->\n<table>                    <!-- Table start -->\n  <thead>                  <!-- Header -->\n    <tr>\n      <th>事業</th>\n      <th>障害の種別</th>\n      <th>本文頁</th>\n      <th>所得制限</th>\n      <th>身体障害者手帳</th>\n    </tr>\n  </thead>\n  <tbody>                  <!-- Body -->\n    <tr>\n      <td></td>\n      <td></td>\n      <td></td>\n      <td></td>\n      <td>視覚障害</td>\n      <td>聴覚又は平衡感覚機能障害</td>\n    </tr>\n    <tr>\n      <td></td>\n      <td></td>\n      <td></td>\n      <td></td>\n      <td>1</td>\n      <td>2</td>\n      <td

In [25]:
res

ChatCompletion(id='chatcmpl-kdu13emces09czw5wdz3cc', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='\n\n```markdown\n# 障害程度別対象事業一覧表\n\n**注釈:**\n- ○：対象\n- △：一部対象\n- 本文参照：詳細は本文を参照のこと\n\n## 概要情報\n\n| 項目 | 内容 |\n| :--- | :--- |\n| **対象範囲** | 身体障害者手帳所持者および特定障害者の福祉サービス |\n| **適用区分** | 障害の種別（視覚、聴覚・平衡感覚など）および等級による |\n\n## 事業一覧詳細表\n\n| 事業名 | ページ番号 | 所得制限 | 視覚障害 (1-4) | 聴覚・平衡感覚 (2-4) |\n| :--- | :---: | :---: | :---: | :---: |\n| **巡回入浴サービス** | 82 | - | - | - |\n| **軽度障害者入浴** | 82 | - | - | 本文参照 |\n| **障害者・児移動支援事業** | 82 | ○ | ○ | - |\n| **重度脳性麻痺者介護事業** | 83 | - | - | - |\n| **在宅心身障害者・児緊急一時介護委託費助成** | 83 | - | - | - |\n| **心身障害者 (児) 短期保護事業** | 84 | ○ | ○ | ○ |\n| **医療的ケア児在宅レスパイト事業** | 85 | - | - | 本文参照 |\n| **居宅訪問型保育事業** | 87 | - | - | 本文参照 |\n| **在宅重症心身障害児 (者) 等訪問事業** | 87 | - | - | 本文参照 |\n| **地域安心生活支援事業** | 88 | - | - | - |\n| **ごみの訪問収集** | 91 | - | - | 本文参照 |\n\n### 災害・緊急時関連事業\n\n| 事業名 | ページ番号 | 所得制限 | 視覚障害 (1-4) | 聴覚・平衡感覚 (2-4) |\n| :---

In [37]:
import base64
from openai import OpenAI

# 1. 画像ファイルをBase64エンコードする関数
def encode_image_to_base64(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

# 2. LM Studioのサーバーを経由してVision推論する関数
def run_lmstudio_vision_inference(image_path, system_prompt, user_text):
    """
    画像ファイルとテキストをLM Studio（Visionモデル）に送信して推論します。
    """
    # 接続設定（ポート1234を指すように変更）
    client = OpenAI(
        base_url="http://127.0.0.1:1234/v1", 
        api_key="lm-studio"
    )

    # パスから画像を読み込んでBase64文字列に変換
    base64_image = encode_image_to_base64(image_path)
    
    # MIMEタイプ（拡張子から推定することもできますが、PNG/JPEG等で一律指定も可能）
    # PDFを画像化したものはPNGとして保存されている想定
    image_url = f"data:image/png;base64,{base64_image}"

    # メッセージの構成（OpenAI Vision APIの標準フォーマット）
    messages = [
        {
            "role": "system", 
            "content": system_prompt
        },
        {
            "role": "user", 
            "content": [
                {"type": "text", "text": user_text},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": image_url
                    }
                }
            ]
        }
    ]

    print(f"Sending image '{image_path}' to LM Studio...")
    
    # 推論リクエスト
    completion = client.chat.completions.create(
        model="local-model",
        messages=messages,
        temperature=0.6,
        # max_tokens=8192  # もし出力が途切れる場合は上限を指定・増やしてください
    )
    
    return completion



In [32]:
# 抽出済み画像のパスを指定 (前のセルで設定したEXTRACT_DATA_DIR配下の画像など)
sample_image_path = EXTRACT_DATA_DIR / "r7syogaisyahukusinotebiki_shusei_page_014.png" 

system_prompt = """あなたはLLMの学習データ作成のエキスパートです。
与えられた画像に記載されている表を、LLMの継続事前学習に適したクリーンなマークダウン形式に変換してください。
"""

user_text = """
この文書の画像を見て、記載されている障害程度別対象事業一覧表の構造を正確に読み取り、
見出し・所得制限・障害種別などを適切に反映した単一のマークダウン表として出力してください。
"""

# 推論の実行
response = run_lmstudio_vision_inference(
    image_path=sample_image_path,
    system_prompt=system_prompt,
    user_text=user_text
)

# 結果の表示
print("\n=== 生成結果 ===")
print(response.choices[0].message.content)


Sending image '/workspace/01_training/01_data_prep/extract_data/r7syogaisyahukusinotebiki_shusei_page_014.png' to LM Studio...

=== 生成結果 ===


# 障害程度別対象事業一覧表 (○対象・△一部対象)

| 事業 | 本文頁 | 所得制限 | 視覚障害<br>(1) | 視覚障害<br>(2) | 視覚障害<br>(3) | 視覚障害<br>(4) | 視覚障害<br>(5) | 視覚障害<br>(6) | 聴覚又は平衡機能障害<br>(2) | 聴覚又は平衡機能障害<br>(3) | 聴覚又は平衡機能障害<br>(4) |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| **東京メトロ旅客運賃の割引** | 120 | | ○ | ○ | ○ | ○ | ○ | ○ | ○ | ○ | ○ |
| **私鉄旅客運賃の割引** | 120 | | ○ | ○ | ○ | ○ | ○ | ○ | ○ | ○ | ○ |
| **タクシー運賃の割引** | 120 | | ○ | ○ | ○ | ○ | ○ | ○ | ○ | ○ | ○ |
| **航空運賃の割引** | 121 | | ○ | ○ | ○ | ○ | ○ | ○ | ○ | ○ | ○ |
| **旅客船運賃の割引** | 121 | | ○ | ○ | ○ | ○ | ○ | ○ | ○ | ○ | ○ |
| **テレビ受信料の減免** | 121 | 有 | △ | △ | △ | △ | △ | △ | △ | △ | △ |
| **水道・下水道料金の減免** | 122 | 有 | \multicolumn{9}{c|}{本文参照} |
| **携帯電話使用料等の割引** | 123 | | ○ | ○ | ○ | ○ | ○ | ○ | ○ | ○ | ○ |
| **郵便料金・ゆうパック運賃等の減免** | 123 | | \multicolumn{9}{c|}{本文参照} |
| **通常郵便葉書 

In [33]:
import time
from pathlib import Path

# --- 1. ディレクトリとプロンプトの設定 ---
# BASE_DIR がすでに定義されている前提です（例: /workspace/01_training/01_data_prep）
# 入力画像フォルダ
input_dir = EXTRACT_DATA_DIR 
# 出力テキストフォルダ（指定の通り extract_tests_qwen3_5）
output_dir = BASE_DIR / "extract_tests_qwen3_5"

# 出力先フォルダが存在しない場合は作成
output_dir.mkdir(parents=True, exist_ok=True)

# すべての画像 (.png / .jpg など) を取得
image_files = sorted(list(input_dir.glob("*.png")) + list(input_dir.glob("*.jpg")))
total_images = len(image_files)

print(f"出力先ディレクトリ: {output_dir}")
print(f"処理対象の画像数: {total_images}枚")

# システムプロンプト（AIの役割）と、ユーザーの指示
system_prompt = """あなたはLLMの学習データ作成のエキスパートです。
与えられた画像に記載されている表やテキストを、LLMの継続事前学習に適したクリーンなマークダウンまたは平文テキスト形式に変換してください。
"""

user_text = """
この文書の画像の構造と内容を正確に読み取り、学習データとして適切なテキスト形式で出力してください。
"""

# --- 2. 一括処理の実行 ---
if total_images > 0:
    total_start_time = time.time()
    
    for i, image_path in enumerate(image_files, 1):
        # 保存するファイル名（拡張子を.mdに変更）
        output_file = output_dir / f"{image_path.stem}.md"
        
        # 既に抽出済みのファイルがある場合はスキップ（処理の再開を容易にするため）
        if output_file.exists():
            print(f"[{i}/{total_images}] スキップ (既に存在します): {output_file.name}")
            continue
            
        print(f"[{i}/{total_images}] 処理中: {image_path.name} ...")
        
        try:
            start_time = time.time()
            
            # 先ほど定義した推論関数を呼び出し
            response = run_lmstudio_vision_inference(
                image_path=image_path,
                system_prompt=system_prompt,
                user_text=user_text
            )
            
            # 抽出された結果を取得
            extracted_text = response.choices[0].message.content
            
            # 結果をテキストファイルに書き込み
            with open(output_file, "w", encoding="utf-8") as f:
                f.write(extracted_text)
                
            elapsed_time = time.time() - start_time
            print(f"  -> 保存完了: {output_file.name} ({elapsed_time:.2f}秒)")
            
        except Exception as e:
            print(f"  -> ❌ 処理エラー {image_path.name}: {e}")
            
    total_elapsed_time = time.time() - total_start_time
    print(f"\nすべての処理が完了しました！ (合計所要時間: {total_elapsed_time:.2f}秒)")
    print(f"結果は {output_dir} を確認してください。")
    
else:
    print("処理対象の画像が見つかりませんでした。input_dir のパスを確認してください。")



出力先ディレクトリ: /workspace/01_training/01_data_prep/extract_tests_qwen3_5
処理対象の画像数: 218枚
[1/218] 処理中: r7syogaisyahukusinotebiki_shusei_page_001.png ...
Sending image '/workspace/01_training/01_data_prep/extract_data/r7syogaisyahukusinotebiki_shusei_page_001.png' to LM Studio...
  -> 保存完了: r7syogaisyahukusinotebiki_shusei_page_001.md (49.80秒)
[2/218] 処理中: r7syogaisyahukusinotebiki_shusei_page_002.png ...
Sending image '/workspace/01_training/01_data_prep/extract_data/r7syogaisyahukusinotebiki_shusei_page_002.png' to LM Studio...
  -> 保存完了: r7syogaisyahukusinotebiki_shusei_page_002.md (93.70秒)
[3/218] 処理中: r7syogaisyahukusinotebiki_shusei_page_003.png ...
Sending image '/workspace/01_training/01_data_prep/extract_data/r7syogaisyahukusinotebiki_shusei_page_003.png' to LM Studio...
  -> 保存完了: r7syogaisyahukusinotebiki_shusei_page_003.md (131.08秒)
[4/218] 処理中: r7syogaisyahukusinotebiki_shusei_page_004.png ...
Sending image '/workspace/01_training/01_data_prep/extract_data/r7syogaisyahukusinotebi

In [34]:
import json
import re
from pathlib import Path

# --- 1. 関数の定義 ---

def clean_markdown_text(text: str) -> str:
    """
    LLMが出力しがちな不要なコードブロック装飾（```markdown 等）を削除し、
    クリーンなテキストに整形します。
    """
    # 先頭の ```markdown や ```プレーンテキストタグを削除
    cleaned = re.sub(r'^```(markdown|text)?\s*\n', '', text, flags=re.IGNORECASE)
    # 末尾の ``` を削除
    cleaned = re.sub(r'\n```\s*$', '', cleaned)
    
    return cleaned.strip()

def process_single_markdown_for_cpt(input_file: Path) -> dict:
    """
    1つのMarkdownファイルを受け取り、クリーンなテキストに整形した上で、
    継続事前学習（CPT）用の構造 {"text": "実際の文章..."} に変換して返します。
    """
    with open(input_file, "r", encoding="utf-8") as f:
        raw_text = f.read()
        
    # クリーニング処理
    clean_text = clean_markdown_text(raw_text)
    
    if not clean_text:
        return None
        
    # JSON構造のレコードを作成
    return {"text": clean_text}



In [35]:
# --- 2. 1ファイルだけでの動作試験 ---

# 試験したいファイルを1つ指定します（ご自身の環境にあるファイル名に合わせてください）
# 例: extracted_texts などのフォルダ内にあるページ
test_input_file = BASE_DIR / "extract_tests_qwen3_5" / "r7syogaisyahukusinotebiki_shusei_page_014.md"
# または EXTRACT_DATA_DIR / "ファイル名" など

# テスト用出力ファイル
test_output_jsonl = BASE_DIR / "test_cpt_dataset.jsonl"

print(f"試験対象ファイル: {test_input_file}")

# ファイルが存在するか確認
if test_input_file.exists():
    
    # 関数を呼び出して1件だけ変換
    cpt_record = process_single_markdown_for_cpt(test_input_file)
    
    if cpt_record:
        # 結果をJSONLの1行として保存
        with open(test_output_jsonl, "w", encoding="utf-8") as outfile:
            # ensure_ascii=False で日本語が \uXXXX に化けるのを防ぐ
            json_line = json.dumps(cpt_record, ensure_ascii=False)
            outfile.write(json_line + "\n")
            
        print(f"\n✅ 変換成功！以下のファイルに保存しました: \n  {test_output_jsonl}")
        
        print("\n--- 変換されたJSONの内容 (先頭200文字) ---")
        print(json_line[:200] + " ...")
    else:
        print("❌ テキストが空だったため変換をスキップしました。")
        
else:
    print(f"❌ 指定されたファイルが見つかりません。パスを確認してください: {test_input_file}")


試験対象ファイル: /workspace/01_training/01_data_prep/extract_tests_qwen3_5/r7syogaisyahukusinotebiki_shusei_page_014.md

✅ 変換成功！以下のファイルに保存しました: 
  /workspace/01_training/01_data_prep/test_cpt_dataset.jsonl

--- 変換されたJSONの内容 (先頭200文字) ---
{"text": "# 障害程度別対象事業一覧表 (○対象・△一部対象)\n\n| 事 業 | 本文頁 | 所得制限 | **身体障害者手帳**<br>視覚障害 (1-6) | | | | | | **聴覚又は平衡感覚機能障害** (2-4) | | |\n| :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---:  ...


In [49]:
system_prompt = """あなたはLLMの高品質な学習データセットを作成するプロフェッショナルです。
入力されるMarkdownテキストには、通常の文章（段落や見出し）と表（テーブル）が混在しています。
LLMが継続事前学習（CPT）において知識の構造を最も理解しやすいよう、以下のルールに従ってテキスト全体を再構成してください。

【厳守するルール】
1. **表形式（Markdownテーブル）は一切使用しない:** 
   表データを発見した場合は、行と列の意味関係を読み取り、「〇〇についての担当は△△（電話: 00-0000）です」のような、主語と述語が明確な自然言語のフルセンテンス、または箇条書きのリストに展開してください。
2. **通常文章の保持:** 
   元からある見出しやパラグラフ（地の文）は、情報を一切削らずにそのまま活かし、表から変換した文章と自然に意味が繋がるように全体の文脈を整えてください。
3. **メタ発言の禁止:** 
   「以下に変換します」「～の文章を作成しました」等の、AI特有の挨拶や前置き、後書きは絶対に出力しないでください。純粋なドキュメントの本文情報のみを出力してください。
4. **自然な日本語での出力の徹底:**
   入力データに存在しないマークダウン表記は絶対しないこと
5. **現在のファイルと見出し等の説明文への埋め込み**
   説明文がどの見出しの中に入っているか、自然な日本語で説明すること
"""

user_text = f"""
以下のMarkdown文書を、ルールに従い「表を含まない自然な文章ベースのドキュメント」に変換してください。

【入力データ（Markdown）】
{markdown_table}
"""

In [50]:
input_markdown_file = BASE_DIR / "extract_tests_qwen3_5" / "r7syogaisyahukusinotebiki_shusei_page_014.md"
if input_markdown_file.exists():
    print(f"[{input_markdown_file.name}] を読み込んで自然語化を開始します...")
    
    # ファイルの中身（Markdownの表）を読み込む
    with open(input_markdown_file, "r", encoding="utf-8") as f:
        markdown_table = f.read()
    # --- プロンプトの設定 ---


    # 推論の実行（画像処理がないので早めに応答が返ります）
    response = run_lmstudio_inference(
        system_prompt=system_prompt,
        user_text=user_text
    )
    
    # 結果の取得と表示
    natural_text_result = response.choices[0].message.content
    
    print("\n================== 変換結果 ==================\n")
    print(natural_text_result)
    
else:
    print(f"❌ ファイルが見つかりません: {input_markdown_file}")


[r7syogaisyahukusinotebiki_shusei_page_014.md] を読み込んで自然語化を開始します...
Generating response via LM Studio...

================== 変換結果 ==================



# 障害程度別対象事業一覧表 (○対象・△一部対象)

本一覧表は、身体障害者手帳を所持する方に対する各種支援事業と、その対象となる障害の等級について記載したものです。まず、交通や生活インフラに関する割引・減免制度から説明します。

東京メトロ旅客運賃の割引については、本文ページ 120 に記載されています。所得制限はなく、身体障害者手帳を持つ視覚障害（等級 1 から 6）および聴覚又は平衡感覚機能障害（等級 2 から 4）の方すべてが対象となります。
私鉄旅客運賃の割引についても、同様に本文ページ 120 に記載されており、所得制限はありません。視覚障害（等級 1 から 6）および聴覚又は平衡感覚機能障害（等級 2 から 4）の方すべてが対象です。
タクシー運賃の割引は、本文ページ 120 に記載されています。所得制限はなく、視覚障害（等級 1 から 6）および聴覚又は平衡感覚機能障害（等級 2 から 4）の方すべてが対象となります。
航空運賃の割引は、本文ページ 121 に記載されています。所得制限はなく、視覚障害（等級 1 から 6）および聴覚又は平衡感覚機能障害（等級 2 から 4）の方すべてが対象です。
旅客船運賃の割引も、本文ページ 121 に記載されており、所得制限はありません。視覚障害（等級 1 から 6）および聴覚又は平衡感覚機能障害（等級 2 から 4）の方すべてが対象となります。

テレビ受信料の減免については、本文ページ 121 に記載されています。所得制限があります。身体障害者手帳を持つ視覚障害（等級 1 から 6）および聴覚又は平衡感覚機能障害（等級 2 から 4）の方については一部対象となります。
水道・下水道料金の減免は、本文ページ 122 に記載されています。所得制限がありますが、詳細な対象範囲については本文を参照してください。
携帯電話使用料等の割引は、本文ページ 123 に記載されています。所得制限はなく、視覚障害（等級 1 から 

In [52]:
import time

# --- 入力元・出力先の設定 ---
input_dir = BASE_DIR / "extract_tests_qwen3_5"
output_dir = BASE_DIR / "cpt_natural_texts"
output_dir.mkdir(parents=True, exist_ok=True)

# --- 対象ファイルの取得とソート ---
md_files = sorted(list(input_dir.glob("*.md")))
total_files = len(md_files)

print(f"入力元: {input_dir}")
print(f"出力先: {output_dir}")
print(f"処理ファイル数: {total_files}件\n")

# --- ループ処理 ---
if total_files > 0:
    total_start_time = time.time()
    
    for i, md_file in enumerate(md_files, 1):
        output_file = output_dir / f"{md_file.stem}.txt"
        
        # 処理済みスキップ
        if output_file.exists():
            print(f"[{i}/{total_files}] スキップ: {output_file.name}")
            continue
            
        print(f"[{i}/{total_files}] 処理中: {md_file.name} ...")
        
        try:
            start_time = time.time()
            
            # ファイル読み込み
            with open(md_file, "r", encoding="utf-8") as f:
                markdown_text = f.read()
                
            # 既存の user_text（指示）に Markdown を結合
            current_prompt = f"{user_text}\n\n【入力データ】\n{markdown_text}"
            
            # 推論実行（関数名と引数はご自身の環境に合わせてください）
            response = run_lmstudio_inference(
                system_prompt=system_prompt,
                user_text=user_text
            )
            
            # 結果の保存
            result_text = response.choices[0].message.content
            with open(output_file, "w", encoding="utf-8") as f:
                f.write(result_text)
                
            elapsed = time.time() - start_time
            print(f"  -> 完了 ({elapsed:.2f}秒)")
            
        except Exception as e:
            print(f"  -> ❌ エラー: {e}")
            
    print(f"\n✅ 全処理完了！ (所要時間: {time.time() - total_start_time:.2f}秒)")
else:
    print("処理対象のMarkdownファイルが見つかりません。")



入力元: /workspace/01_training/01_data_prep/extract_tests_qwen3_5
出力先: /workspace/01_training/01_data_prep/cpt_natural_texts
処理ファイル数: 218件

[1/218] 処理中: r7syogaisyahukusinotebiki_shusei_page_001.md ...
Generating response via LM Studio...
  -> 完了 (208.09秒)
[2/218] 処理中: r7syogaisyahukusinotebiki_shusei_page_002.md ...
Generating response via LM Studio...
  -> 完了 (208.64秒)
[3/218] 処理中: r7syogaisyahukusinotebiki_shusei_page_003.md ...
Generating response via LM Studio...
  -> 完了 (190.92秒)
[4/218] 処理中: r7syogaisyahukusinotebiki_shusei_page_004.md ...
Generating response via LM Studio...
  -> 完了 (221.13秒)
[5/218] 処理中: r7syogaisyahukusinotebiki_shusei_page_005.md ...
Generating response via LM Studio...
  -> 完了 (223.02秒)
[6/218] 処理中: r7syogaisyahukusinotebiki_shusei_page_006.md ...
Generating response via LM Studio...
  -> 完了 (146.39秒)
[7/218] 処理中: r7syogaisyahukusinotebiki_shusei_page_007.md ...
Generating response via LM Studio...
  -> 完了 (250.33秒)
[8/218] 処理中: r7syogaisyahukusinotebiki_shuse

最後に学習用データとして1ファイルにまとめる

In [53]:
import json
from pathlib import Path

# --- 1. 入力ディレクトリと出力ファイルの設定 ---
input_dir = BASE_DIR / "cpt_natural_texts"
output_file = BASE_DIR / "cpt_input.jsonl"  # 拡張子は .jsonl が標準です

print(f"入力ディレクトリ: {input_dir}")
print(f"出力ファイル名: {output_file}")

# --- 2. ファイルの取得とソート ---
# 抽出されたすべてのテキスト(.txt)を取得し、ファイル名順（ページ順）にソート
txt_files = sorted(list(input_dir.glob("*.txt")))
total_files = len(txt_files)

print(f"処理対象のテキスト数: {total_files}件\n")

# --- 3. 結合と学習データ（JSONL）化 ---
if total_files > 0:
    processed_count = 0
    
    # 出力ファイルを書き込みモードで開く
    with open(output_file, "w", encoding="utf-8") as outfile:
        
        for txt_file in txt_files:
            try:
                # 1ファイル（1ページ分）のテキストを読み込む
                with open(txt_file, "r", encoding="utf-8") as infile:
                    content = infile.read().strip()
                
                # 中身が空でなければ処理
                if content:
                    # 継続事前学習の標準フォーマットである {"text": "..."} という辞書を作成
                    record = {
                        "text": content
                    }
                    
                    # 辞書をJSON文字列化（ensure_ascii=Falseで日本語を維持）
                    json_line = json.dumps(record, ensure_ascii=False)
                    
                    # 1ファイルを1行のJSONとしてファイルに書き込み
                    outfile.write(json_line + "\n")
                    processed_count += 1
                    
            except Exception as e:
                print(f"❌ エラー ({txt_file.name}): {e}")
                
    print(f"✅ 処理完了！ {processed_count} 件のドキュメントを結合し、学習データを作成しました。")
    print(f"最終出力: {output_file}")
    
    # --- プレビュー表示（正しく1行に収まっているか確認） ---
    if processed_count > 0:
        print("\n--- 📝 生成されたデータのプレビュー（最初の1行目の書き出し） ---")
        with open(output_file, "r", encoding="utf-8") as f:
            first_line = f.readline()
            preview = first_line[:150] + " ..." if len(first_line) > 150 else first_line
            print(preview)
            
else:
    print("❌ 処理対象のテキストファイルが見つかりません。input_dir を確認してください。")



入力ディレクトリ: /workspace/01_training/01_data_prep/cpt_natural_texts
出力ファイル名: /workspace/01_training/01_data_prep/cpt_input.jsonl
処理対象のテキスト数: 218件

✅ 処理完了！ 218 件のドキュメントを結合し、学習データを作成しました。
最終出力: /workspace/01_training/01_data_prep/cpt_input.jsonl

--- 📝 生成されたデータのプレビュー（最初の1行目の書き出し） ---
{"text": "# 障害程度別対象事業一覧表 (○対象・△一部対象)\n\n以下は、障害程度別の対象となる事業の一覧です。\n\n- **東京メトロ旅客運賃の割引**（本文頁：120）視覚障害等級 1 から 6、聴覚又は平衡感覚機能障害等級 2 から 4 のすべてが対象となります。所得制限はあり ...


# 全データのSFT向けQA生成


In [29]:
import json
import re
import time
from pathlib import Path

# =========================================================
# 1. プロンプトの外部定義（変数化）
# =========================================================

QA_SYSTEM_PROMPT = """あなたは有能なAIアシスタントのファインチューニング（SFT）用学習データを作成する専門家です。
与えられたMarkdownドキュメントから、ユーザーの指示や質問（instruction）と、それに対するAIのアシスタントの回答（output）のペアを作成してください。

【厳守事項】
1. 合計で「ちょうど35件」のQ&Aペアを作成してください。
2. 35件のうち「25件」は、ドキュメントの内容に完全に基づき、書かれている事実を回答とする正確なQ&Aにしてください。
3. 35件のうち「5件」は、ドキュメントの内容に完全に基づき、なぜその結果となるのかの質問とその回答となるようなQ&Aにしてください。
4. 35件のうち「4件」は、ドキュメントの内容に完全に基づき、逆引きとなるようなQ&Aにしてください。
5. 35件のうち「残り1件」は、ドキュメントに意図的に「記載されていない情報（架空の内容や関連外の質問）」のQを作成し、Aを「申し訳ありませんが、提供された資料の中にその情報は見当たりません。」という、存在しないことを正しく指摘する回答にしてください。
6. 出力は必ず以下のJSON配列フォーマットのみとし、Markdownのコードブロック（```json ... ```）などの装飾タグや説明文、前置きは一切出力しないでください。
7. 質問の言い回しが被らないように、「〜について教えて」「〜は必要ですか？」「〜の有無を確認したい」などの文末を活用してください。
8. 回答の言い回しは、質問を受けた職員が市民に回答する言い回しで生成してください。
9. ドキュメントに記載されていない内容は絶対に生成しないでください。推測もダメです。

【出力JSONフォーマット】
[
  {"instruction": "〇〇について説明してください。", "output": "〇〇は～です。"},
  ...
  {"instruction": "ドキュメントに存在しない質問", "output": "申し訳ありませんが、提供された資料の中にその情報は見当たりません。"}
]
"""

# LLMに投げる際の具体的なテキストテンプレート
def get_user_text(markdown_text: str) -> str:
    return f"""以下のドキュメントから、条件を厳密に満たすSFT用Q&Aデータを抽出し、JSONの配列形式だけで出力してください。

【ドキュメント】
{markdown_text}
"""





In [7]:
# =========================================================
# 2. 1ファイルを対象にしたQA生成関数の定義
# =========================================================

def generate_sft_qa_for_file(md_file_path: Path) -> list:
    """
    1つのMarkdownファイルから、LLMを使用してSFT用QAのリストを生成します。
    """
    with open(md_file_path, "r", encoding="utf-8") as f:
        markdown_text = f.read()

    # テキスト専用推論関数を呼び出す（引数名はご自身の環境に合わせてください）
    response = run_lmstudio_inference(
        system_prompt=QA_SYSTEM_PROMPT,
        user_text=get_user_text(markdown_text)
    )
    
    result_text = response.choices[0].message.content.strip()
    
    # LLMが不意に付けた ```json や ``` を綺麗に削除（JSONパースエラー防止）
    result_text = re.sub(r'^```(?:json)?\s*\n', '', result_text, flags=re.IGNORECASE)
    result_text = re.sub(r'\n```\s*$', '', result_text)
    
    try:
        # 文字列をPythonのリスト（辞書の配列）にパースして返す
        qa_list = json.loads(result_text)
        return qa_list
    except json.JSONDecodeError as e:
        print(f"⚠️ JSONパースエラー: {md_file_path.name}")
        print(f"出力テキスト先頭: {result_text[:100]}...")
        return []




In [12]:
input_dir = BASE_DIR / "../01_data_prep/extract_tests_qwen3_5"
output_jsonl = BASE_DIR / "../01_data_prep/sft_data_generated.jsonl"  # お好きな名前に変更してください

print(input_dir)

/home/hiroooo000-remote/dev/ft-test/01_training/02_SFT/../01_data_prep/extract_tests_qwen3_5


In [30]:
# =========================================================
# 3. 全ファイル対象のループ処理
# =========================================================

# ディレクトリ内の Markdownファイル を取得してソート
all_md_files = sorted(list(input_dir.glob("*.md")))
target_files = []

# 「_010.mdより前のファイルは表紙や目次のため無視する」フィルタリング処理
for file_path in all_md_files:
    # ファイル名からページ番号(例: 010)を抽出
    match = re.search(r'page_(\d+)\.md$', file_path.name)
    if match:
        page_num = int(match.group(1))
        # 10以上のページのみ処理対象（1〜9はスキップ）
        if page_num == 10:
            target_files.append(file_path)

total_files = len(target_files)
print(f"処理対象ファイル数: {total_files}件 (ページ010以降のファイルを対象)\n")

# 一気に処理を開始
if total_files > 0:
    total_start_time = time.time()
    total_generated_qa = 0
    
    # 出力ファイルを初期化（上書き開始）
    with open(output_jsonl, "w", encoding="utf-8") as f:
        pass 
        
    for i, md_file in enumerate(target_files, 1):
        print(f"[{i}/{total_files}] 抽出中: {md_file.name} ...")
        
        start_time = time.time()
        
        # 関数を呼び出して1ファイル分のQAセット(25件)を取得
        qa_list = generate_sft_qa_for_file(md_file)
        
        if qa_list:
            # 取得したQAセットをJSONLファイルに1件(1行)ずつ書き込む
            with open(output_jsonl, "a", encoding="utf-8") as outfile:
                for qa in qa_list:
                    json_line = json.dumps(qa, ensure_ascii=False)
                    outfile.write(json_line + "\n")
            
            elapsed = time.time() - start_time
            print(f"  -> {len(qa_list)}件 生成完了 ({elapsed:.2f}秒)")
            total_generated_qa += len(qa_list)
        else:
            print("  -> ❌ QAの生成に失敗しました（JSONフォーマット不正など）")

    total_time = time.time() - total_start_time
    print(f"\n✅ すべてのSFTデータ生成が完了しました！ (合計 {total_generated_qa} 件のQAデータ)")
    print(f"出力データ: {output_jsonl}")

else:
    print("条件を満たす Markdownファイル が見つかりませんでした。")

処理対象ファイル数: 1件 (ページ010以降のファイルを対象)

[1/1] 抽出中: r7syogaisyahukusinotebiki_shusei_page_010.md ...
Generating response via LM Studio...
  -> 35件 生成完了 (60.90秒)

✅ すべてのSFTデータ生成が完了しました！ (合計 35 件のQAデータ)
出力データ: /home/hiroooo000-remote/dev/ft-test/01_training/02_SFT/../01_data_prep/sft_data_generated.jsonl
